In [2]:
import boto3

In [ ]:
# Configurações do DataLake
S3_ENDPOINT_URL = "https://sdadasdasdasdasdasdasda.storage.supabase.co/storage/v1/s3"
AWS_REGION = "us-east-1"
AWS_ACCESS_KEY_ID = "asdasdasdwdwwqdqwdqdqw"
AWS_SECRET_ACCESS_KEY = "asdasdwqdqdqwasdadsadqw"
BUCKET_NAME = "ecommerce_data"

In [4]:
# Criar cliente S3
s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [5]:
# Listar arquivos no bucket
response = s3.list_objects(Bucket=BUCKET_NAME)
arquivos = [obj["Key"] for obj in response["Contents"]]

In [6]:
print(arquivos)

['clientes.parquet', 'preco_competidores.parquet', 'produtos.parquet', 'vendas.parquet']


In [15]:
import pandas as pd

In [21]:
# Baixar arquivo Parquet
FILE_KEY = "vendas.parquet"
response = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_KEY)
parquet_bytes = response["Body"].read()

In [27]:
# Converter Parquet para DataFrame
# Converter Parquet para DataFrame
import io                         # garante que está definido
import pyarrow.parquet as pq      # importa o sub‑módulo

buf = io.BytesIO(parquet_bytes)   # cria um buffer a partir dos bytes lidos
table = pq.read_table(buf)        # lê o parquet com pyarrow
df_vendas = table.to_pandas()     # converte para pandas

# opcional: conferir que deu certo
df_vendas.head()

,id_venda,data_venda,id_cliente,id_produto,canal_venda,quantidade,preco_unitario
0,sal_adff6978b0c6,2025-12-13 17:38:09+00:00,cus_508a15ccf0fa,prd_96fbc500aec1,loja_fisica,2,64.79
1,sal_0648f0d17798,2025-12-13 10:32:35+00:00,cus_f64907d41a69,prd_686e2aab873d,ecommerce,1,89.50
2,sal_2cd165121973,2025-12-13 11:55:39+00:00,cus_78ab1cf16e12,prd_223c9b6a0a76,loja_fisica,2,69.69
3,sal_0cdb7d67970c,2025-12-13 20:48:10+00:00,cus_2b1b3e2a1515,prd_30518d908a92,ecommerce,1,80.99
4,sal_0ed16afae5b1,2025-12-13 22:15:13+00:00,cus_644bf6820c61,prd_ff35b88df534,ecommerce,1,190.71


In [ ]:
# ============================================
# PASSO 2: Salvar no PostgreSQL
# ============================================

# Configurações do PostgreSQL (Supabase)
# Usar postgresql+psycopg2:// ao invés de postgresql://
DATABASE_URL = "postgresql+psycopg2://postgres:dasdasdasd"

In [33]:
from sqlalchemy import create_engine

In [35]:
# Criar engine de conexão
engine = create_engine(DATABASE_URL)

In [ ]:
# Salvar DataFrame em tabela PostgreSQL
# if_exists: 'replace' (substitui), 'append' (adiciona), 'fail' (erro se existir)
df_vendas.to_sql(
    "vendas",  # Nome da tabela
    engine,  # Engine de conexão
    if_exists="replace",  # Substituir se existir
    index=False  # Não salvar índice
)

In [36]:
# Ler dados salvos para verificar
df_verificacao = pd.read_sql_query("SELECT * FROM vendas", engine)

In [38]:
# ============================================
# OUTRAS OPERAÇÕES COM PANDAS E SQL
# ============================================

# Executar query e trazer para pandas
query = """
SELECT
    COUNT(*) as total_vendas,
    SUM(quantidade) as total_quantidade
FROM vendas
"""
df_agregado = pd.read_sql_query(query, engine)

# Fechar conexão
engine.dispose()

In [39]:
df_agregado

,total_vendas,total_quantidade
0,3020,4322.0
